## **La Corsa alle Rinnovabili**
*Un'analisi globale di 79 paesi su transizione energetica, dipendenza dai fossili e disuguaglianze nel consumo pro capite*

Leggo il dataset online -> questo serve per evitare di dover caricare ogni volta il file nella directory

In [ ]:
import pandas as pd

# Carico il dataset da GitHub (senza doverlo poi caricare nella directory ogni volta, lo prendiamo dal link)
url = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"
df = pd.read_csv(url)

print("Righe:", df.shape[0])
print("Colonne:", df.shape[1])

df.head()

Righe: 23377
Colonne: 130


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN


Controllo quali paesi non unna un codice ISO per eliminarli da dataset (pulizia)

In [ ]:
# Vediamo quali "country" NON hanno un codice ISO -> sono aggregati, non paesi veri
finti = df[df["iso_code"].isna()]["country"].unique()
print("Righe che NON sono paesi reali:", len(finti))
print(finti)

Righe che NON sono paesi reali: 94
['ASEAN (Ember)' 'Africa' 'Africa (EI)' 'Africa (EIA)' 'Africa (Ember)'
 'Africa (Shift)' 'Asia' 'Asia (Ember)' 'Asia Pacific (EI)'
 'Asia and Oceania (EIA)' 'Asia and Oceania (Shift)'
 'Australia and New Zealand (EIA)' 'CIS (EI)' 'Central America (EI)'
 'Central and South America (EIA)' 'Central and South America (Shift)'
 'Czechoslovakia' 'EU (Ember)' 'EU28 (Shift)' 'East Germany'
 'Eastern Africa (EI)' 'Eastern Europe and Eurasia (EIA)' 'Eurasia (EIA)'
 'Eurasia (Shift)' 'Europe' 'Europe (EI)' 'Europe (EIA)' 'Europe (Ember)'
 'Europe (Shift)' 'European Union (27)' 'G20 (Ember)' 'G7 (Ember)'
 'High-income countries' 'Kosovo' 'Latin America and Caribbean (Ember)'
 'Low-income countries' 'Lower-middle-income countries'
 'Middle Africa (EI)' 'Middle East (EI)' 'Middle East (EIA)'
 'Middle East (Ember)' 'Middle East (Shift)' 'Non-OECD (EI)'
 'Non-OECD (EIA)' 'Non-OPEC (EI)' 'Non-OPEC (EIA)' 'North America'
 'North America (EI)' 'North America (Ember)' '

In [ ]:
# Prendiamo la lista dei "finti paesi" trovata prima, ma togliamo Kosovo (è un paese vero senza codice ISO)
da_rimuovere = [p for p in finti if p != "Kosovo"]

# Teniamo solo le righe il cui "country" NON è nella lista da rimuovere
df_pulito = df[~df["country"].isin(da_rimuovere)]

print("Righe prima:", df.shape[0])
print("Righe dopo:", df_pulito.shape[0])
print("Paesi rimasti (esempio):", df_pulito["country"].unique()[:10])

Righe prima: 23377
Righe dopo: 17291
Paesi rimasti (esempio): ['Afghanistan' 'Albania' 'Algeria' 'American Samoa' 'Angola' 'Antarctica'
 'Antigua and Barbuda' 'Argentina' 'Armenia' 'Aruba']


Gestisco solo i dati dal 2000 in poi (il dataset parte dal 1900 ma con dati pocco rilevanti).

Salvo i dati per creare un nuovo file e lavorare da quello

In [ ]:
df_pulito = df_pulito[df_pulito["year"] >= 2000]

print("Righe dopo il filtro anni:", df_pulito.shape[0])
print("Anno minimo:", df_pulito["year"].min())
print("Anno massimo:", df_pulito["year"].max())

Righe dopo il filtro anni: 5575
Anno minimo: 2000
Anno massimo: 2025


Andiamo a vedere chi ha troppi dati mancanti su renewables_share_energy (la colonna che useremo di più, quindi conviene controllarla per prima)

In [ ]:
# Per ogni paese, contiamo quante righe hanno il dato mancante su questa colonna
mancanti_per_paese = df_pulito.groupby("country")["renewables_share_energy"].apply(lambda x: x.isna().sum())

# Quante righe totali abbiamo per paese (dal 2000 in poi dovrebbero essere circa 24-26 per ognuno)
totale_per_paese = df_pulito.groupby("country")["renewables_share_energy"].size()

# Mettiamo le due cose insieme in una tabella per confrontarle facilmente
riepilogo = pd.DataFrame({"mancanti": mancanti_per_paese, "totale": totale_per_paese})
riepilogo["% mancante"] = (riepilogo["mancanti"] / riepilogo["totale"] * 100).round(1)

# Ordiniamo dal peggiore (più dati mancanti) al migliore
riepilogo = riepilogo.sort_values("% mancante", ascending=False)

riepilogo.head(20)

,mancanti,totale,% mancante
country,,,
Afghanistan,25,25,100.0
Albania,25,25,100.0
American Samoa,25,25,100.0
Angola,25,25,100.0
Antarctica,24,24,100.0
Armenia,26,26,100.0
Antigua and Barbuda,25,25,100.0
Aruba,25,25,100.0
Belize,25,25,100.0


La colonna renewables_share_energy è calcolata a partire da statistiche energetiche complesse (quelle del "Energy Institute", ex BP), che storicamente coprono solo le economie più grandi (circa 80-90 paesi principali) e non tutti i ~200 paesi/territori del mondo. Per questo tantissimi paesi piccoli (Antigua, Bahamas, Bhutan, Antarctica...) risultano al 100% vuoti — semplicemente per loro quel dato non viene calcolato da nessuno.

Vediamo la situazione generale: quanti paesi hanno dati completi e quanti sono invece messi male:

In [ ]:
# Contiamo quanti paesi rientrano in fasce di "qualità dati"
fasce = pd.cut(riepilogo["% mancante"], bins=[-1, 0, 25, 50, 75, 100],
                labels=["0% (completo)", "1-25%", "26-50%", "51-75%", "76-100%"])

print(fasce.value_counts().sort_index())

% mancante
0% (completo)     11
1-25%             68
26-50%             0
51-75%             0
76-100%          142
Name: count, dtype: int64


Questa distribuzione è proprio quello che ci aspettavamo, e conferma che non è un problema di dati mancanti "a caso" ma di copertura strutturale: o un paese viene tracciato dalle fonti energetiche principali (e allora ha quasi tutti i dati), oppure non viene tracciato affatto (e ha quasi tutto vuoto) — infatti non c'è nessun paese nelle fasce intermedie (26-50%, 51-75%), è tutto o bianco o nero.

Questo rende la scelta della soglia facile: teniamo i paesi con al massimo il 25% di dati mancanti (gli 11 + 68 = 79 paesi delle prime due fasce), e togliamo i 142 che sono sostanzialmente "non tracciati". Questi 79 sono comunque le economie principali del mondo — più che sufficienti per un'analisi seria, anzi probabilmente sono esattamente i paesi più interessanti da confrontare.

In [ ]:
# Prendiamo solo i paesi con al massimo il 25% di dati mancanti su renewables_share_energy
paesi_da_tenere = riepilogo[riepilogo["% mancante"] <= 25].index

df_pulito = df_pulito[df_pulito["country"].isin(paesi_da_tenere)]

print("Paesi rimasti:", df_pulito["country"].nunique())
print("Righe totali dopo la pulizia:", df_pulito.shape[0])

Paesi rimasti: 79
Righe totali dopo la pulizia: 2043


Ora salviamolo su Google Drive, così la prossima volta che apriamo il notebook non dobbiamo rifare tutta questa pulizia — carichi direttamente il file pulito.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Creiamo una cartella dedicata al progetto (se non esiste già)
import os
cartella = "/content/drive/MyDrive/progetto_energia"
os.makedirs(cartella, exist_ok=True)

# Salviamo il dataset pulito dentro
df_pulito.to_csv(f"{cartella}/energia_pulito.csv", index=False)

print("Salvato in:", f"{cartella}/energia_pulito.csv")

Salvato in: /content/drive/MyDrive/progetto_energia/energia_pulito.csv


In [ ]:
df_pulito = pd.read_csv("/content/drive/MyDrive/progetto_energia/energia_pulito.csv")
df_pulito

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,Algeria,2000,DZA,30903894.0,2.085542e+11,NaN,0.0,0.0,0.0,0.000,...,0.000,0.000,NaN,0.000,0.000,0.000,0.00,0.000,0.000,0.000
1,Algeria,2001,DZA,31331226.0,2.234203e+11,NaN,0.0,0.0,0.0,0.000,...,0.000,0.000,NaN,0.000,0.000,0.000,0.00,0.000,0.000,0.000
2,Algeria,2002,DZA,31750832.0,2.453609e+11,NaN,0.0,0.0,0.0,0.000,...,0.000,0.000,NaN,0.000,0.000,0.000,0.00,0.000,0.000,0.000
3,Algeria,2003,DZA,32175813.0,2.735686e+11,NaN,0.0,0.0,0.0,0.000,...,0.000,0.000,NaN,0.000,0.000,0.000,0.00,0.000,0.000,0.000
4,Algeria,2004,DZA,32628286.0,2.967736e+11,NaN,0.0,0.0,0.0,0.000,...,0.000,0.000,NaN,0.000,0.000,0.000,0.00,0.000,0.000,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2038,Vietnam,2021,VNM,98935101.0,7.719120e+11,NaN,0.0,0.0,0.0,3.234,...,10.311,5.338,240.224,5.804,8.229,33.760,3.34,83.176,1.317,0.682
2039,Vietnam,2022,VNM,99680656.0,8.338039e+11,NaN,0.0,0.0,0.0,3.812,...,9.713,4.967,172.104,14.108,22.337,91.191,9.09,224.082,3.429,1.753
2040,Vietnam,2023,VNM,100352189.0,NaN,NaN,0.0,0.0,0.0,8.669,...,9.288,4.617,27.445,5.922,28.259,115.493,11.59,281.594,4.190,2.082
2041,Vietnam,2024,VNM,100987685.0,NaN,NaN,0.0,0.0,0.0,10.199,...,8.518,4.329,10.021,2.832,31.090,126.253,12.75,307.862,4.200,2.134


In [ ]:
df = df_pulito.copy()

# **1. Chi sta vincendo la corsa alle rinnovabili?**

Analizziamo la quota di energia rinnovabile raggiunta oggi dai vari paesi e come si è evoluta nel tempo per i leader del settore.

In [ ]:
import plotly.express as px

ultimo_anno = (
    df[df["renewables_share_energy"].notna()]
    ["year"]
    .max()
)


df_last = (
    df[(df['year'] == ultimo_anno) &
       (df['renewables_share_energy'] >= 25)]
    .dropna(subset=['renewables_share_energy'])
    .sort_values(by='renewables_share_energy', ascending=True)
    .reset_index(drop=True)
)

def assegna_colore(valore):
    if valore >= 70:
        return "#2F5D50"
    elif valore >= 50:
        return "#52796F"
    elif valore >= 30:
        return "#84A98C"
    else:
        return "#B7CDB5"

df_last["colore"] = df_last["renewables_share_energy"].apply(assegna_colore)
df_last["country_label"] = df_last["country"] + "  —"

fig = px.bar(
    df_last,
    x="renewables_share_energy",
    y="country_label",
    orientation="h",
    text="renewables_share_energy",
    color="colore",
    color_discrete_map="identity"
)

fig.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    textfont=dict(
        size=12,
        color="#555555"),
    marker_line_width=0,
    width=0.55
)

fig.update_layout(
    title={
        "text": f"<b>Chi guida la transizione energetica ({ultimo_anno})</b>",
        "x":0.05,
        "xanchor":"left"
    },
    xaxis_title="Quota di energia rinnovabile (%)",
    yaxis_title="",
    template="plotly_white",
    width=1000,
    height=550,
    showlegend=False,
    margin=dict(
        l=120,
        r=120,
        t=80,
        b=60
    )
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.12)"
)

fig.update_yaxes(
    showgrid=False,
    categoryorder="array",
    categoryarray=df_last["country_label"]
)

fig.show()

*Nota: i paesi in cima alla classifica sono spesso Stati piccoli con condizioni geografiche particolarmente favorevoli (risorse idroelettriche o geotermiche abbondanti rispetto alla popolazione da servire) — la % di rinnovabili premia quindi anche la "fortuna geografica", non solo lo sforzo di transizione. Per un grande paese industrializzato, spostare la stessa percentuale richiede investimenti assoluti molto più ingenti.*

Islanda e Norvegia dominano nettamente, superando il 70% di energia rinnovabile — un risultato legato alla grande disponibilità di risorse idroelettriche e geotermiche. La maggior parte degli altri paesi si concentra invece tra il 15% e il 40%.

Interessante notare che la classifica attuale non racconta tutta la storia: la **Danimarca**, pur non comparendo tra i primissimi, è il paese che ha guadagnato più terreno in assoluto, con un incremento di circa **+35 punti percentuali dal 2000** — seguita da Portogallo e Lituania.

In [ ]:
import plotly.express as px

# Paesi leader
paesi_leader = ["Iceland", "Norway", "Sweden", "Brazil", "Portugal", "Austria"]

trend = (
     df[
        (df["country"].isin(paesi_leader)) &
        (df["year"] >= 2000)
    ]
    .dropna(subset=["renewables_share_energy"])
    .sort_values(["country", "year"])
)


palette_verde = [
    "#2F5D50",  # verde scuro morbido
    "#52796F",
    "#84A98C",
    "#A4C3B2",
    "#CAD2C5"
]


fig = px.line(
    trend,
    x="year",
    y="renewables_share_energy",
    color="country",
    markers=True,
    color_discrete_sequence=palette_verde
)


fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=7)
)


fig.update_layout(

    title={
        "text": "<b>Evoluzione della quota di energia rinnovabile nei paesi leader</b>",
        "x":0.05
    },

    xaxis_title="Anno",
    yaxis_title="Quota di energia rinnovabile (%)",

    template="plotly_white",

    width=1000,
    height=600,

    hovermode="x unified",

    legend_title="Paesi"
)


fig.show()

*Quindi, chi sta vincendo la corsa alle rinnovabili?*

Non c'è una risposta unica: Islanda e Norvegia dominano oggi grazie a risorse naturali uniche, ma la vera storia della transizione la scrivono paesi come Danimarca e Portogallo, che partendo da zero hanno costruito la crescita più rapida — a dimostrazione che la leadership energetica si misura tanto nel presente quanto nel percorso compiuto.

# **2. L'italia come si posiziona rispetto agli altri grandi paesi europeri?**

L'analisi continua con l'evoluzione della quota di energia rinnovabile in Italia rispetto ai principali paesi europei (Germania, Francia e Spagna) nel corso degli anni. L'obiettivo è valutare il posizionamento dell'Italia nella transizione energetica europea attraverso un confronto diretto dei dati.

A livello mondiale, l’Italia risulta 38ª, evidenziando un ritardo significativo rispetto ai paesi leader della transizione energetica.

In [ ]:
countries = ["Italy", "France", "Spain", "Germany"]

df[df["country"].isin(countries)] \
    .groupby("country")["year"] \
    .agg(["min", "max", "count"])

,min,max,count
country,,,
France,2000,2025,26
Germany,2000,2025,26
Italy,2000,2025,26
Spain,2000,2025,26


In [ ]:
import plotly.express as px

countries = ["Italy", "Germany", "France", "Spain"]

df_europe = df[
    (df["country"].isin(countries)) &
    (df["year"]>=2000)
    ]

colori_paesi = {
    "Italy": "#1B4332",
    "Germany": "#74A57F",
    "France": "#95C99A",
    "Spain": "#B7D7B0"
}

fig = px.line(
    df_europe,
    x="year",
    y="renewables_share_energy",
    color="country",
    markers=True,
    color_discrete_map=colori_paesi
)

fig.update_traces(
    line=dict(width=2.5),
    marker=dict(size=7)
)

# Evidenzio l'Italia
fig.update_traces(
    selector=dict(name="Italy"),
    line=dict(width=4),
    marker=dict(size=9)
)

fig.update_layout(
    title={
        "text": "<b>Evoluzione della quota di energia rinnovabile: Italia vs grandi paesi europei</b>",
        "x":0.05,
        "xanchor":"left"
    },

    xaxis_title="Anno",
    yaxis_title="Quota di energia rinnovabile (%)",

    template="plotly_white",

    width=1000,
    height=550,

    hovermode="x unified",

    legend_title="Paesi",

    margin=dict(
        l=70,
        r=80,
        t=80,
        b=60
    )
)

fig.update_xaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.12)"
)

fig.update_yaxes(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.12)"
)

fig.show()

L'Italia mostra una crescita costante nel tempo, portandosi su livelli competitivi rispetto ai principali paesi europei. Tuttavia, il confronto evidenzia come Germania e soprattutto Spagna abbiano accelerato maggiormente negli ultimi anni, mentre la Francia rimane su valori inferiori.

# **3. Da cosa dipendiamo ancora? Il mix energetico**

Dopo aver visto come i paesi leader stiano guidando la transizione e come l’Italia, pur collocandosi al 38° posto a livello mondiale, mantenga comunque una buona posizione rispetto ai principali paesi europei, il passo successivo è capire da cosa dipendiamo ancora. Il mix energetico rivela infatti quali fonti continuano a sostenere i sistemi nazionali: un punto di verità che mostra quanto ogni paese sia ancora legato alle proprie scelte storiche e quanto spazio reale resti per accelerare la trasformazione.

In [ ]:
[c for c in df.columns if "share_energy" in c]

['biofuel_share_energy',
 'coal_share_energy',
 'electricity_share_energy',
 'fossil_share_energy',
 'gas_share_energy',
 'hydro_share_energy',
 'low_carbon_share_energy',
 'nuclear_share_energy',
 'oil_share_energy',
 'other_renewables_share_energy',
 'renewables_share_energy',
 'solar_share_energy',
 'wind_share_energy']

In [ ]:
import plotly.express as px

countries = [
    "Iceland",
    "Norway",
    "Brazil",
    "France",
    "Germany",
    "Italy",
    "China",
    "India",
    "Spain",
    "Austria",
    "Switzerland"
]

df_mix = df[
    (df["country"].isin(countries)) &
    (df["year"] == ultimo_anno)
].copy()


fonti = {
    "renewables_share_energy": "Rinnovabili",
    "nuclear_share_energy": "Nucleare",
    "gas_share_energy": "Gas",
    "oil_share_energy": "Petrolio",
    "coal_share_energy": "Carbone"
}


mix = df_mix[
    ["country"] + list(fonti.keys())
].melt(
    id_vars="country",
    var_name="Fonte",
    value_name="Quota"
)


mix["Fonte"] = mix["Fonte"].map(fonti)


colori_mix = {
    "Rinnovabili": "#74C69D",
    "Nucleare": "#C0C6CC",
    "Gas": "#E9C46A",
    "Petrolio": "#6D6875",
    "Carbone": "#2F3B45"
}


fig = px.bar(
    mix,
    x="country",
    y="Quota",
    color="Fonte",
    text="Quota",
    color_discrete_map=colori_mix
)


fig.update_traces(
    texttemplate="%{text:.0f}%",
    textposition="inside"
)


fig.update_layout(
    barmode="stack",

    title={
        "text": f"<b>Da cosa dipende ancora il sistema energetico? ({ultimo_anno})</b>",
        "x":0.05,
        "xanchor":"left"
    },

    xaxis_title="Paese",
    yaxis_title="Quota del mix energetico (%)",

    template="plotly_white",

    width=1000,
    height=600,

    legend_title="Fonte energetica",

    margin=dict(
        l=70,
        r=80,
        t=80,
        b=60
    )
)


fig.update_yaxes(
    range=[0,100],
    showgrid=True,
    gridcolor="rgba(0,0,0,0.12)"
)


fig.show()

Il grafico rivela che la dipendenza energetica dei paesi resta profondamente diversa: alcuni, come Islanda e Norvegia, hanno ormai un sistema quasi interamente fondato sulle rinnovabili, mentre altri continuano a poggiare su fonti fossili o sul nucleare. È una fotografia immediata di quanto la transizione non proceda in modo uniforme: ogni paese avanza con velocità e fragilità proprie, e il mix energetico racconta esattamente dove ciascuno è rimasto indietro e da cosa dipende ancora.

In [ ]:
fonti = {
    "renewables_share_energy": "Rinnovabili",
    "nuclear_share_energy": "Nucleare",
    "gas_share_energy": "Gas",
    "oil_share_energy": "Petrolio",
    "coal_share_energy": "Carbone"
}

dipendenza_media = (
    df[df["year"] == ultimo_anno][list(fonti.keys())]
    .mean()
    .rename(index=fonti)
    .sort_values(ascending=False)
    .round(1)
    .reset_index()
)
dipendenza_media.columns = ["Fonte", "Quota media globale (%)"]
dipendenza_media

,Fonte,Quota media globale (%)
0,Petrolio,38.8
1,Gas,27.4
2,Rinnovabili,17.4
3,Carbone,12.2
4,Nucleare,4.1



A livello globale, la dipendenza maggiore resta dal **petrolio** (38,8% in media), seguito dal **gas** (27,4%) — insieme, le due fonti fossili coprono ancora quasi i due terzi del fabbisogno energetico mondiale. Le **rinnovabili**, pur essendo cresciute molto negli ultimi anni (come visto nei grafici precedenti), si fermano in media al 17,4%: meno della metà del solo petrolio. Il **nucleare** resta la fonte più marginale (4,1%), nonostante sia a basse emissioni — segno di quanto sia rimasto un tema politicamente ed economicamente controverso in molti paesi.

Questo dato aiuta a leggere in prospettiva i risultati dei grafici precedenti: anche i paesi "leader" nella transizione (Islanda, Norvegia) restano eccezioni rispetto a un quadro globale ancora fortemente dipendente dai combustibili fossili.

# **4. Chi cresce più in fretta tra solare ed eolico?**

Dopo aver visto da cosa dipendono ancora i sistemi energetici, il passo successivo è capire quali tecnologie stanno davvero trainando la transizione.

In [ ]:
import plotly.express as px

# Media per anno, su tutti i paesi puliti, di solare ed eolico
andamento = (
    df.groupby("year")[["solar_share_energy", "wind_share_energy"]]
    .mean()
    .reset_index()
    .melt(id_vars="year", var_name="Fonte", value_name="Quota media (%)")
    .dropna(subset=["Quota media (%)"])   # <-- aggiunta: toglie gli anni senza dati (es. 2025)
)

andamento["Fonte"] = andamento["Fonte"].map({
    "solar_share_energy": "Solare",
    "wind_share_energy": "Eolico"
})

colori_fonti = {
    "Solare": "#F4A261",
    "Eolico": "#457B9D"
}

fig = px.line(
    andamento,
    x="year",
    y="Quota media (%)",
    color="Fonte",
    markers=True,
    color_discrete_map=colori_fonti
)

fig.update_traces(line=dict(width=3), marker=dict(size=7))

fig.update_layout(
    title={
        "text": "<b>Chi cresce più in fretta: solare o eolico?</b>",
        "x": 0.05,
        "xanchor": "left"
    },
    xaxis_title="Anno",
    yaxis_title="Quota media di energia (%)",
    template="plotly_white",
    width=1000,
    height=550,
    hovermode="x unified",
    legend_title="Fonte"
)

fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.12)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.12)")

fig.show()

In [ ]:
inizio = andamento[andamento["year"] == andamento["year"].min()]
fine = andamento[andamento["year"] == andamento["year"].max()]

for fonte in ["Solare", "Eolico"]:
    v_inizio = inizio[inizio["Fonte"] == fonte]["Quota media (%)"].values[0]
    v_fine = fine[fine["Fonte"] == fonte]["Quota media (%)"].values[0]
    crescita = v_fine / v_inizio if v_inizio > 0 else float("inf")
    print(f"{fonte}: da {v_inizio:.2f}% a {v_fine:.2f}% (x{crescita:.1f})")

Solare: da 0.00% a 2.74% (x3280.6)
Eolico: da 0.11% a 3.48% (x30.4)


Guardando la crescita in termini assoluti, le due fonti hanno guadagnato terreno in modo abbastanza simile: l'eolico è passato dallo 0,11% al 3,48% (+3,4 punti percentuali), il solare dallo 0,00% al 2,74% (+2,7 punti percentuali).

Il dato più interessante è però *da dove* sono partite: l'eolico era già una tecnologia minimamente diffusa nel 2000, mentre il solare partiva praticamente da zero — **la sua crescita**, seppur numericamente vicina a quella dell'eolico, **rappresenta quindi una tecnologia nata quasi dal nulla nell'arco di 25 anni, sostenuta dal crollo dei costi dei pannelli fotovoltaici nell'ultimo decennio**.

Solare ed eolico sono le fonti rinnovabili di cui si parla di più, ma non sono le uniche — e soprattutto non sono ancora le più utilizzate in assoluto. Per avere un quadro completo, vediamo come si posizionano rispetto a tutte le altre fonti rinnovabili.

In [ ]:
import plotly.express as px

fonti_rinnovabili = {
    "hydro_share_energy": "Idroelettrico",
    "solar_share_energy": "Solare",
    "wind_share_energy": "Eolico",
    "biofuel_share_energy": "Bioenergie",
    "other_renewables_share_energy": "Altre rinnovabili"
}

rinnovabili_media = (
    df[df["year"] == ultimo_anno][list(fonti_rinnovabili.keys())]
    .mean()
    .rename(index=fonti_rinnovabili)
    .sort_values(ascending=True)   # ascending perché il grafico è orizzontale
    .round(2)
    .reset_index()
)
rinnovabili_media.columns = ["Fonte", "Quota media globale (%)"]

colori_rinnovabili = {
    "Idroelettrico": "#2A9D8F",
    "Solare": "#F4A261",
    "Eolico": "#457B9D",
    "Bioenergie": "#8D6A4B",
    "Altre rinnovabili": "#B0B0B0"
}

fig = px.bar(
    rinnovabili_media,
    x="Quota media globale (%)",
    y="Fonte",
    orientation="h",
    text="Quota media globale (%)",
    color="Fonte",
    color_discrete_map=colori_rinnovabili
)

fig.update_traces(
    texttemplate="%{text:.2f}%",
    textposition="outside",
    marker_line_width=0
)

fig.update_layout(
    title={
        "text": f"<b>Quali rinnovabili contano davvero? ({ultimo_anno})</b>",
        "x": 0.05,
        "xanchor": "left"
    },
    xaxis_title="Quota media di energia (%)",
    yaxis_title="",
    template="plotly_white",
    width=1000,
    height=450,
    showlegend=False,
    margin=dict(l=140, r=100, t=80, b=60)
)

fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.12)")

fig.show()

Il grafico conferma che, nonostante la crescita rapida vista nel grafico precedente, solare (2,74%) ed eolico (3,48%) restano ancora dietro l'**idroelettrico**, che con l'8,10% è di gran lunga la fonte rinnovabile più diffusa a livello globale — più del doppio dell'eolico e quasi tre volte il solare. Bioenergie (0,79%) e altre rinnovabili minori (2,30%) restano marginali.

È un dato che ridimensiona la narrazione comune: la "rivoluzione" delle rinnovabili raccontata dai media è soprattutto una rivoluzione di *crescita* (solare ed eolico stanno accelerando più di ogni altra fonte), ma in termini di *volumi assoluti* la transizione energetica poggia ancora, oggi, sull'idroelettrico — una tecnologia matura, spesso poco raccontata perché non è "nuova".

# **5. Consumo energetico pro capite: i paesi ricchi consumano davvero di più?**

Confrontiamo la ricchezza media per abitante (PIL pro capite) con quanta energia consuma in media una singola persona in ciascun paese, per capire se esiste davvero una relazione tra le due cose.

In [ ]:
import plotly.express as px

# Per ogni paese, prendiamo l'ultima riga con TUTTI e 3 i valori disponibili
# (gdp si ferma al 2022, energy_per_capita arriva al 2024: non possiamo forzare lo stesso anno per tutti)
df_gdp = (
    df.dropna(subset=["gdp", "energy_per_capita", "population"])
    .sort_values("year")
    .groupby("country")
    .tail(1)
    .copy()
)

# gdp nel dataset è il PIL TOTALE del paese: lo dividiamo per la popolazione
# per ottenere il PIL pro capite, unità confrontabile con energy_per_capita
df_gdp["gdp_per_capita"] = df_gdp["gdp"] / df_gdp["population"]

print("Paesi rimasti nel grafico:", df_gdp["country"].nunique())

# Etichetta visibile solo per i paesi più popolosi (i "giganti demografici")
soglia_popolazione = df_gdp["population"].sort_values(ascending=False).iloc[4]  # top 5
df_gdp["etichetta"] = df_gdp.apply(
    lambda r: r["country"] if r["population"] >= soglia_popolazione else "",
    axis=1
)

fig = px.scatter(
    df_gdp,
    x="gdp_per_capita",
    y="energy_per_capita",
    size="population",
    hover_name="country",
    text="etichetta",
    log_x=True,
    trendline="ols",
    color_discrete_sequence=["#40916C"]   # verde coerente con gli altri grafici
)

fig.update_traces(
    marker=dict(opacity=0.65, line=dict(width=0.5, color="white")),
    textposition="top center",
    textfont=dict(size=11, color="#2F3B45")
)

fig.update_layout(
    title={
        "text": "<b>Più ricchi, più energivori? (dati più recenti disponibili per paese)</b>",
        "x": 0.05,
        "xanchor": "left"
    },
    xaxis_title="PIL pro capite (scala logaritmica, $)",
    yaxis_title="Consumo di energia pro capite (kWh)",
    template="plotly_white",
    width=1000,
    height=600
)

fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.12)")
fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.12)")

fig.show()

correlazione = df_gdp["gdp_per_capita"].corr(df_gdp["energy_per_capita"])
print(f"Correlazione tra PIL pro capite ed energia pro capite: {correlazione:.2f}")

Paesi rimasti nel grafico: 79


Correlazione tra PIL pro capite ed energia pro capite: 0.78


Più un paese è ricco, più energia consuma ogni suo abitante — e non in modo lineare, ma sempre più velocemente man mano che sale il reddito (la correlazione è di **0,78**, un legame forte).

Attenzione però a non confondere due cose diverse: **quanti abitanti ha un paese** e **quanta energia consuma ciascuno di loro**. Cina, India, Indonesia e Pakistan hanno insieme miliardi di persone e pesano tantissimo sul consumo energetico *mondiale*, ma preso singolarmente ogni abitante consuma ancora relativamente poco, perché il reddito medio è ancora contenuto. Gli Stati Uniti invece hanno sia tanta popolazione *sia* un reddito alto, quindi ogni cittadino consuma molta energia.

I pochi paesi che spiccano **sopra** la linea di tendenza sono piccoli Stati (spesso legati al petrolio o a un'industria pesante): pochi abitanti, ma un consumo energetico a testa molto alto rispetto alle loro dimensioni.

**Il messaggio di fondo**: la ricchezza spinge il consumo di energia a persona, ma il consumo energetico *mondiale* nel suo complesso dipende soprattutto da quanta gente c'è sulla Terra — sono due leve diverse che si sommano.

# **La transazione energetica vista dal mondo**

In [ ]:
import plotly.express as px
import pandas as pd

df_mappa = (
    df.dropna(subset=["renewables_share_energy", "iso_code"])
    .sort_values("year")
    .groupby("country")
    .tail(1)
    .copy()
)

bins = [0, 15, 30, 50, 100]
etichette = ["0-15%", "15-30%", "30-50%", "50%+"]
df_mappa["fascia"] = pd.cut(df_mappa["renewables_share_energy"], bins=bins, labels=etichette)

colori_fasce = {
    "0-15%": "#D8F3DC",
    "15-30%": "#95D5B2",
    "30-50%": "#52B788",
    "50%+": "#1B4332"
}

fig = px.choropleth(
    df_mappa,
    locations="iso_code",
    color="fascia",
    hover_name="country",
    hover_data={"renewables_share_energy": ":.1f", "fascia": False, "iso_code": False},
    color_discrete_map=colori_fasce,
    category_orders={"fascia": etichette}
)

fig.update_traces(marker_line_color="#B0B0B0", marker_line_width=0.6)

fig.update_layout(
    title={
        "text": "<b>La transizione energetica vista dal mondo, oggi</b>",
        "x": 0.03,
        "xanchor": "left",
        "font": dict(size=22, color="#222222")
    },
    template="plotly_white",
    width=1000,
    height=600,
    geo=dict(
        showframe=False,
        showcoastlines=False,
        showland=True,
        landcolor="white",          # paesi senza dati: bianco, come nel tuo esempio
        showocean=True,
        oceancolor="white",         # niente blu, sfondo bianco pulito
        showlakes=False,
        showcountries=True,
        countrycolor="#B0B0B0",     # bordi sottili grigi, ben visibili
        projection_type="natural earth",
        bgcolor="rgba(0,0,0,0)"
    ),
    paper_bgcolor="rgba(0,0,0,0)",
    legend=dict(
        orientation="h",            # legenda orizzontale
        yanchor="bottom", y=-0.1,
        xanchor="center", x=0.5,
        title=""
    )
)

fig.show()

# Conclusioni

Mettendo insieme le cinque analisi, il quadro che emerge non è "il mondo si sta convertendo alle rinnovabili" né "il mondo dipende ancora troppo dai fossili" — sono vere entrambe le cose insieme, a seconda di dove si guarda.

**La transizione è reale ma profondamente diseguale.** Islanda e Norvegia superano il 70% grazie a risorse naturali che pochissimi altri paesi hanno (geotermico, idroelettrico); Danimarca e Portogallo dimostrano che una crescita rapida è possibile anche senza quel vantaggio geografico. Ma la mappa finale lo conferma a colpo d'occhio: la maggior parte del mondo, soprattutto Asia, Africa e Medio Oriente, è ancora lontana da quote significative.

**A livello globale, i fossili restano il vero pilastro.** Petrolio e gas coprono da soli quasi i due terzi del mix energetico medio mondiale (38,8% + 27,4%), contro il 17,4% delle rinnovabili. Anche i paesi "leader" nella classifica sono eccezioni statistiche, non la norma.

**Tra le rinnovabili, la vera storia è la crescita, non ancora il volume.** Solare ed eolico stanno accelerando più di ogni altra fonte, ma restano dietro all'idroelettrico (8,1% contro 2,7% e 3,5%) — la "rivoluzione verde" raccontata dai media è soprattutto un fenomeno di velocità di crescita, non ancora di peso reale sul totale.

**Ricchezza e popolazione tirano in direzioni diverse.** Il consumo energetico pro capite è legato in modo forte alla ricchezza di un paese (correlazione 0,78) e cresce più che proporzionalmente superata una certa soglia di reddito. Ma i giganti demografici (Cina, India, Indonesia, Pakistan) mostrano che avere tanti abitanti non equivale ad avere alti consumi pro capite — sono due dimensioni del problema energetico che vanno lette separatamente.

**L'Italia si colloca in una posizione intermedia**: 38ª a livello mondiale, con una crescita costante ma più lenta rispetto a Germania e Spagna, che negli ultimi anni hanno accelerato di più.

In sintesi: non esiste "un" mondo della transizione energetica, ma tanti percorsi diversi — guidati da geografia, ricchezza e scelte politiche — che si muovono a velocità molto differenti verso lo stesso obiettivo.